In [ ]:
!pip install captum

     |████████████████████████████████| 5.7MB 3.9MB/s 


In [ ]:
import numpy as np
import pandas as pd
from os import listdir
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn import functional as F
import torch.nn as nn
import torch.backends.cudnn as cudnn
import torchvision
import torchvision.transforms as transforms
from torch.autograd import Variable
import torch.optim as optim
from torch.autograd import Function
from torchvision import models
from torchvision import utils
import sys
import pickle

import matplotlib
import matplotlib.pyplot as plt
import scipy
import scipy.ndimage as ndimage
import matplotlib.patches as patches
import pdb


import numpy as np
from os import listdir
import skimage.transform
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn import functional as F
import torch.nn as nn
import torch.backends.cudnn as cudnn
import torchvision
import torchvision.transforms as transforms
from torch.autograd import Variable
import torch.optim as optim
from torch.autograd import Function
from torchvision import models
from torchvision import utils
import cv2
import sys
import os
import pickle
from collections import defaultdict
from collections import OrderedDict

import skimage
from skimage.io import *
from skimage.transform import *

import scipy
import scipy.ndimage as ndimage
import scipy.ndimage.filters as filters
from scipy.ndimage import binary_dilation
import matplotlib.patches as patches

In [ ]:
%cd drive/MyDrive/project1
!pwd

/content/drive/MyDrive/project1
/content/drive/MyDrive/project1


In [ ]:
class DenseNet121(nn.Module):
    """Model modified.
    The architecture of our model is the same as standard DenseNet121
    except the classifier layer which has an additional sigmoid function.
    """
    def __init__(self, out_size):
        super(DenseNet121, self).__init__()
        self.densenet121 = torchvision.models.densenet121(pretrained=True)
        num_ftrs = self.densenet121.classifier.in_features
        self.densenet121.classifier = nn.Sequential(
            nn.Linear(num_ftrs, out_size),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.densenet121(x)
        return x


In [ ]:
model = DenseNet121(8).cuda()
model = torch.nn.DataParallel(model)
state_dict = torch.load("denseNet.pkl") # the best model
model.load_state_dict(state_dict,strict=False)
model.eval()
print("model loaded")


Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth



model loaded


In [ ]:
test_X = []
print("load and transform image")
#for i in range(len(test_list)):
#image_path = os.path.join(img_folder_path, test_list[i])
import imageio
img = imageio.imread('00000147_001.png')
#img = scipy.misc.imred('1.png')
if img.shape != (1024,1024):
  img = img[:,:,0]
img_resized = skimage.transform.resize(img,(256,256))
test_X.append((np.array(img_resized)).reshape(256,256,1))
#if i % 100==0:
#print(i)
test_X = np.array(test_X)

load and transform image


In [ ]:
class ChestXrayDataSet_plot(Dataset):
	def __init__(self, input_X = test_X, transform=None):
		self.X = np.uint8(test_X*255)
		self.transform = transform

	def __getitem__(self, index):
		"""
		Args:
		    index: the index of item
		Returns:
		    image
		"""
		current_X = np.tile(self.X[index],3)
		image = self.transform(current_X)
		return image
	def __len__(self):
		return len(self.X)

test_dataset = ChestXrayDataSet_plot(input_X = test_X,transform=transforms.Compose([
                                        transforms.ToPILImage(),
                                        transforms.CenterCrop(224),
                                        transforms.ToTensor(),
                                        transforms.Normalize([0.485, 0.456, 0.406],[0.229, 0.224, 0.225])
                                        ]))

In [ ]:

input_img = Variable((test_dataset[0]).unsqueeze(0).cuda(), requires_grad=True)

In [ ]:
model.forward(input_img)

tensor([[0.4834, 0.1226, 0.5823, 0.5521, 0.1455, 0.2232, 0.5240, 0.1717]],
       device='cuda:0', grad_fn=<SigmoidBackward>)

In [ ]:
output = model(input_img)

In [ ]:
output

tensor([[0.4834, 0.1226, 0.5823, 0.5521, 0.1455, 0.2232, 0.5240, 0.1717]],
       device='cuda:0', grad_fn=<SigmoidBackward>)

In [ ]:
import torch
#import torch.nn.functional as F
import torch.nn.functional as F

from PIL import Image

import os
import json
import numpy as np
from matplotlib.colors import LinearSegmentedColormap

import torchvision
from torchvision import models
from torchvision import transforms

from captum.attr import IntegratedGradients
from captum.attr import GradientShap
from captum.attr import Occlusion
from captum.attr import NoiseTunnel
from captum.attr import visualization as viz

In [ ]:
output = F.softmax(output, dim=1)

In [ ]:
output

tensor([[0.1403, 0.0978, 0.1548, 0.1502, 0.1000, 0.1081, 0.1461, 0.1027]],
       device='cuda:0', grad_fn=<SoftmaxBackward>)

In [ ]:
prediction_score, pred_label_idx = torch.topk(output, 1)

In [ ]:
#pred_label_idx.squeeze_()
integrated_gradients = IntegratedGradients(model)

In [ ]:
attributions_ig = integrated_gradients.attribute(input_img,target=1, n_steps=100)



In [ ]:
torch.cuda.empty_cache()
!rm -rf ~/.nv

In [ ]:
default_cmap = LinearSegmentedColormap.from_list('custom blue',
                                                 [(0, '#ffffff'),
                                                  (0.25, '#000000'),
                                                  (1, '#000000')], N=256)

In [ ]:
_ = viz.visualize_image_attr(np.transpose(attributions_ig.squeeze().cpu().detach().numpy(), (1,2,0)),
                             np.transpose(input_img.squeeze().cpu().detach().numpy(), (1,2,0)),
                             method='heat_map',
                             cmap=default_cmap,
                             show_colorbar=True,
                             sign='positive',
                             outlier_perc=1)